<a href="https://colab.research.google.com/github/Koushikgmurthy/AI-LAB/blob/main/GAlab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-chroma \
    langchain-huggingface \
    pypdf \
    sentence-transformers \
    chromadb \
    streamlit \
    pyngrok

In [11]:
import os

folders = [
    "data",
    "data/documents",
    "chroma_db"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Project folders created")

for folder in folders:
    print("📁", folder)

✅ Project folders created
📁 data
📁 data/documents
📁 chroma_db


In [14]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(
        filename,
        f"data/documents/{filename}"
    )

print("\n✅ PDFs uploaded:")
print(os.listdir("data/documents"))

Saving 3rd sem syllabus.pdf to 3rd sem syllabus.pdf

✅ PDFs uploaded:
['LPC17xx.dbgconf.base@0.0.0', '3rd sem syllabus.pdf']


In [15]:
import os

pdf_files = [
    f for f in os.listdir("data/documents")
    if f.lower().endswith(".pdf")
]

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("📄", pdf)

Number of PDFs: 1
📄 3rd sem syllabus.pdf


In [16]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader(
    "data/documents"
)

documents = loader.load()

print("Total pages loaded:", len(documents))

Total pages loaded: 38


In [17]:
if len(documents) == 0:
    raise ValueError("❌ No PDF documents were loaded.")

total_characters = sum(
    len(doc.page_content)
    for doc in documents
)

print("Total extracted characters:", total_characters)

print("\nFirst page preview:")
print(documents[0].page_content[:1000])

Total extracted characters: 69998

First page preview:
2024-25 The National Institute of Engineering  
Department of Information Science & Engineering 
Code: BCS302    Course: Digital Design & Computer Organization  
Credits: 4     CIE: 50 Marks 
L:T:P - 3:0:2     SEE: 50 Marks  
SEE Hours: 3     Total Marks:100  
 
Prerequisites if any Fundamentals of Logic 
Learning objectives 
1. To provide the knowledge to explain the fundamentals of Logic circuits and 
combinational circuits.  
2. To introduce the design of sequential circuit systems, Registers and Counters  
3. The basics involved in number representation and arithmetic operations in the 
computer system. 
4. Basic processor concept, instruction execution and Bus architecture, Memory 
architecture and mapping techniques.  
 
Course Outcomes: 
On the successful completion of the course, the student will be able to  
COs Course Outcomes 
CO1 Use logic minimization techniques and design combinational circuits using logic gates. 
CO2

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 123


In [19]:
for chunk in chunks:

    source = chunk.metadata.get("source", "Unknown")

    chunk.metadata["file_name"] = os.path.basename(source)

    if "page" in chunk.metadata:
        chunk.metadata["page_number"] = chunk.metadata["page"] + 1
    else:
        chunk.metadata["page_number"] = "Unknown"

print("✅ Metadata added")

✅ Metadata added


In [20]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded


In [21]:
test_embedding = embeddings.embed_query(
    "What is a database?"
)

print("Embedding dimensions:", len(test_embedding))

Embedding dimensions: 384


In [22]:
import shutil
import os

if os.path.exists("chroma_db"):
    shutil.rmtree("chroma_db")

os.makedirs("chroma_db", exist_ok=True)

print("✅ ChromaDB folder prepared")

✅ ChromaDB folder prepared


In [23]:
from langchain_chroma import Chroma

db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

print("✅ All PDF chunks stored in ChromaDB")

InternalError: Error updating collection: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [25]:
import os
import shutil

# Delete old Chroma database
chroma_path = "/content/chroma_db"

if os.path.exists(chroma_path):
    shutil.rmtree(chroma_path)

# Create a fresh directory
os.makedirs(chroma_path, exist_ok=True)

print("✅ Old ChromaDB removed")
print("✅ New ChromaDB created")
print("Path:", chroma_path)

✅ Old ChromaDB removed
✅ New ChromaDB created
Path: /content/chroma_db


In [26]:
print("Documents:", len(documents))
print("Chunks:", len(chunks))

Documents: 38
Chunks: 123


In [27]:
test_embedding = embeddings.embed_query(
    "What is a database?"
)

print("Embedding size:", len(test_embedding))

Embedding size: 384


In [28]:
from langchain_chroma import Chroma

db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/chroma_db",
    collection_name="pdf_rag"
)

print("✅ ChromaDB created successfully!")

✅ ChromaDB created successfully!


In [29]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

question = "What is a primary key?"

results = retriever.invoke(question)

print("Retrieved:", len(results))

for i, doc in enumerate(results):

    print("\n" + "=" * 60)
    print("Result:", i + 1)

    print(
        "File:",
        doc.metadata.get("file_name", "Unknown")
    )

    print(
        "Page:",
        doc.metadata.get("page_number", "Unknown")
    )

    print("\nText:")
    print(doc.page_content[:500])

Retrieved: 4

Result: 1
File: 3rd sem syllabus.pdf
Page: 25

Text:
Properties, Deleting Properties, Testing Properties, Enumerating Properties, 
Extending Objects, Serializing Objects, Object Methods 
2 - 1 
3.2 
Arrays: Creating Arrays, Reading and Writing Array Elements, Sparse 
Arrays, Array Length, Adding and Deleting Array Elements, Iterating Arrays, 
Multidimensional Arrays, Array Methods, Array-Like Objects, Strings as 
Arrays 
2 - 1 
3.3 
Functions: Defining Functions, Invoking Functions, Function Arguments and 
Parameters, Functions as Values, Function

Result: 2
File: 3rd sem syllabus.pdf
Page: 35

Text:
CO2 Explain Installation procedure and basic commands in GiT. 
 
Mapping with POs and PSOs: 
COs PO1 PO2 PO3 PO4 PO5 PO6 PO7 PO8 PO9 PO10 PO11 PO12 
 
PSO1 PSO2 
CO1 3 3 3 3 3 2 1 2 3 3 3 2 3 3 
CO2 3 3 3 3 3 3 1 1 2 2 3 2 3 2 
Mapping Strength:  Strong– 3 Medium – 2 Low – 1

Result: 3
File: 3rd sem syllabus.pdf
Page: 11

Text:
list. The operations to be supported are: 
i) In

In [30]:
def create_prompt(question, documents):

    context = ""

    for doc in documents:
        file_name = doc.metadata.get("file_name", "Unknown")
        page = doc.metadata.get("page_number", "Unknown")

        context += f"""
SOURCE: {file_name}
PAGE: {page}

{doc.page_content}

-------------------------
"""

    prompt = f"""
You are a PDF-based AI assistant.

Answer the user's question using ONLY the information
provided in the PDF context below.

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. If the answer is not available in the PDF context,
   say exactly:
   "The answer is not available in the uploaded PDFs."
4. Give a clear and simple answer.
5. At the end, mention the source PDF and page number.

PDF CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    return prompt

In [31]:
question = "What is a primary key?"

results = retriever.invoke(question)

prompt = create_prompt(
    question,
    results
)

print(prompt)


You are a PDF-based AI assistant.

Answer the user's question using ONLY the information
provided in the PDF context below.

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. If the answer is not available in the PDF context,
   say exactly:
   "The answer is not available in the uploaded PDFs."
4. Give a clear and simple answer.
5. At the end, mention the source PDF and page number.

PDF CONTEXT:

SOURCE: 3rd sem syllabus.pdf
PAGE: 25

Properties, Deleting Properties, Testing Properties, Enumerating Properties, 
Extending Objects, Serializing Objects, Object Methods 
2 - 1 
3.2 
Arrays: Creating Arrays, Reading and Writing Array Elements, Sparse 
Arrays, Array Length, Adding and Deleting Array Elements, Iterating Arrays, 
Multidimensional Arrays, Array Methods, Array-Like Objects, Strings as 
Arrays 
2 - 1 
3.3 
Functions: Defining Functions, Invoking Functions, Function Arguments and 
Parameters, Functions as Values, Functions as Namespaces, Closures, 
Functi

In [32]:
!pip install -q -U langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [33]:
!pip install -q -U langchain-google-genai

In [37]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass(
    "Enter your Gemini API key: "
)

Enter your Gemini API key: ··········


In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("✅ Gemini loaded")


✅ Gemini loaded


In [39]:
response = llm.invoke(
    "Explain what a primary key is in one sentence."
)

print(response.content)

A primary key is a column or set of columns that uniquely identifies each record in a database table and cannot contain null values.


In [40]:
def ask_pdf(question):

    # Retrieve relevant documents
    results = retriever.invoke(question)

    # Create prompt
    prompt = create_prompt(
        question,
        results
    )

    # Ask Gemini
    response = llm.invoke(prompt)

    return response.content, results

In [41]:
answer, sources = ask_pdf(
    "What is a primary key?"
)

print(answer)

The answer is not available in the uploaded PDFs.


In [42]:
print("\n📚 SOURCES")
print("=" * 50)

for doc in sources:

    file_name = doc.metadata.get(
        "file_name",
        "Unknown"
    )

    page = doc.metadata.get(
        "page_number",
        "Unknown"
    )

    print(f"📄 {file_name} — Page {page}")


📚 SOURCES
📄 3rd sem syllabus.pdf — Page 25
📄 3rd sem syllabus.pdf — Page 35
📄 3rd sem syllabus.pdf — Page 11
📄 3rd sem syllabus.pdf — Page 21


In [43]:
answer, sources = ask_pdf(
    "Explain normalization."
)

print(answer)

print("\nSources:")

for doc in sources:
    print(
        f"📄 {doc.metadata.get('file_name')} "
        f"— Page {doc.metadata.get('page_number')}"
    )

The answer is not available in the uploaded PDFs.

Sources:
📄 3rd sem syllabus.pdf — Page 18
📄 3rd sem syllabus.pdf — Page 17
📄 3rd sem syllabus.pdf — Page 8
📄 3rd sem syllabus.pdf — Page 17


In [44]:
answer, sources = ask_pdf(
    "What are the different types of normalization?"
)

print(answer)

The answer is not available in the uploaded PDFs.


In [45]:
answer, sources = ask_pdf(
    "Who won the FIFA World Cup in 1986?"
)

print(answer)

The answer is not available in the uploaded PDFs.


In [ ]:
while True:

    question = input(
        "\n📚 Ask your PDF "
        "(type 'exit' to stop): "
    )

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer, sources = ask_pdf(question)

    print("\n🤖 ANSWER")
    print("=" * 60)
    print(answer)

    print("\n📚 SOURCES")
    print("=" * 60)

    shown_sources = set()

    for doc in sources:

        file_name = doc.metadata.get(
            "file_name",
            "Unknown"
        )

        page = doc.metadata.get(
            "page_number",
            "Unknown"
        )

        source = f"{file_name} — Page {page}"

        if source not in shown_sources:
            print("📄", source)
            shown_sources.add(source)


📚 Ask your PDF (type 'exit' to stop): what are the 3 sem syllabus

🤖 ANSWER
The 3rd sem syllabus includes the following courses:

*   **Social Connect & Responsibilities** (Course Code: BSCK307)
*   **Data Structures and Applications** (Course Code: BCS304)
*   **Digital Design & Computer Organization** (Course Code: BCS302)

Source: 3rd sem syllabus.pdf, Page: 1, 7, 28

📚 SOURCES
📄 3rd sem syllabus.pdf — Page 28
📄 3rd sem syllabus.pdf — Page 7
📄 3rd sem syllabus.pdf — Page 1
📄 3rd sem syllabus.pdf — Page 3

📚 Ask your PDF (type 'exit' to stop): what are all the subjects included in the syllabus?

🤖 ANSWER
The subjects included in the syllabus are:
*   Digital Design & Computer Organization
*   Social Connect & Responsibilities
*   Data Structures and Applications

SOURCE: 3rd sem syllabus.pdf (Page: 1, 7, 28)

📚 SOURCES
📄 3rd sem syllabus.pdf — Page 1
📄 3rd sem syllabus.pdf — Page 3
📄 3rd sem syllabus.pdf — Page 28
📄 3rd sem syllabus.pdf — Page 7

📚 Ask your PDF (type 'exit' to stop)